# Pipeline

## Problema

Tenemos nuestro modelo en producción. Nuestro API de Flask carga nuestro modelo, y tenemos un endpoint para enviar datos y realizar alguna predicción con éstos:

```python
curl -X POST http://127.0.0.1:5000/predecir -d '{"input": [0,1,0.6159084,0,0,0.55547282,1]}
```

Lamentablemetne, nuestro API no es el más práctico del mundo. Para empezar, el formato en el que pide los datos es muy poco amigable. El JSON de entrada podría ser algo así:

```python
{
  "Pclass": 0,
  "Sex": 1,
  "Age": 0.6159084,
  "SibSp": 0,
  "Parch": 0,
  "Fare": 0.55547282,
  "Embarked": 1,
}
```

Esto estaría mucho mejor porque mínimo sabríamos los valores que estamos asignando a cada feature. No obstante, ¿Qué significa una edad de 0.6159084?

Recordemos que en nuestro proceso de limpieza y feature engineering, realizamos transformaciones significativas a los datos: conversión a variables numéricas, normalización y escalamiento. Por supuesto que no podemos esperar que los usuarios de nuestro API realicen estas transformaciones por su propia cuenta. 

El objetivo final de esta implementación será que podamos usar nuestro API con un JSON de entrada como el siguiente:

```python
{
  "Pclass": 2,
  "Sex": "male",
  "Age": 46,
  "SibSp": 0,
  "Parch": 0,
  "Fare": 7.2500,
  "Embarked": "C",
}
```

Es decir, sin transformaciones. Podríamos creer que implementaremos lógica en nuestro API para realizar transformaciones antes de hacer inferencia, pero la implementación que seguiremos es mucho más simple y elegante que eso. 

Lo que haremos a continuación es utilizar el objeto Pipeline de Scikit Learn, el cual, como su nombre lo sugiere, creará las directrices que permitirán el flujo de datos desde su entrada hasta su salida en forma de precciones. 

## Importación de librerías

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import ColumnTransformer 
from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline 
from sklearn.pipeline import Pipeline 

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas de evaluación
from sklearn.metrics import accuracy_score


# Para guardar el modelo
import pickle

## Carga de datos

In [5]:
df = pd.read_csv('./data/titanic_clean.csv')

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## Preprocesamiento

A continuación crearemos un objeto especial que usaremos en nuestro pipeline. Este objeto definirá los pasos de preprocesamiento que se ejecutarán tanto en la fase de entrenamiento como en la fase de inferencia.

In [6]:
# Definir preprocesamiento
preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(), ['Sex', 'Embarked']),  # OneHotEncoder para columnas categóricas
        ('age', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Age']),
        ('fare', QuantileTransformer(output_distribution='normal', n_quantiles=500), ['Fare'])
    ],
    remainder='passthrough'  # Mantener otras columnas sin cambios
)

## Definición de modelos

In [7]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'model__C': [0.01, 0.1, 1, 10, 100],
            'model__penalty': ['l1', 'l2'],
            'model__solver': ['liblinear', 'saga'],
            'model__max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'model__kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'model__C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'model__splitter': ['best', 'random'],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4],
            'model__max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    'Clasificador XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3]
        }
    },
    'Clasificador LGBM': {
        'modelo': LGBMClassifier(),
        'parametros': {
            'model__n_estimators': [10, 100],
            'model__max_depth': [None, 1, 2, 3],
            'model__learning_rate': [0.1, 0.2, 0.3],
            'model__verbose': [-1]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'model__alpha': [0.1, 1.0, 10.0]
        }
    }
}

## División de datos

In [9]:
# División de datos
X = df.drop(['Survived'], axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=100)

# Imprimir dimensiones
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (712, 7)
y_train: (712,)
X_test: (179, 7)
y_test: (179,)


## Entrenamiento

In [11]:
# Inicializar variables para almacenar los puntajes de los modelos y el mejor estimador
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    # Crear pipeline para el modelo
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('scaler', MinMaxScaler()),  # MinMaxScaler se aplica a todas las columnas después del preprocesamiento
        ('model', info_modelo['modelo'])  # Modelo placeholder
    ])
    
    # Inicializar GridSearchCV con el pipeline y los hiperparámetros
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=info_modelo['parametros'],
        cv=5,
        scoring='accuracy',
        verbose=0,
        n_jobs=-1,
    )

    # Ajustar GridSearchCV con los datos de entrenamiento
    grid_search.fit(X_train, y_train)
    
    # Hacer predicciones con el modelo ajustado
    y_pred = grid_search.predict(X_test)
    
    # Calcular la precisión de las predicciones
    precision = accuracy_score(y_test, y_pred)
    
    # Almacenar los resultados del modelo
    puntajes_modelos.append({
        'Modelo': nombre,
        'Precisión': precision
    })

    estimadores[nombre] = grid_search.best_estimator_
    
    # Actualizar el mejor modelo si la precisión actual es mayor que la mejor precisión encontrada
    if precision > mejor_precision:
        mejor_modelo = nombre
        mejor_precision = precision
        mejor_estimador = grid_search.best_estimator_

# Inicializar GridSearchCV con el pipeline y los hiperparámetros
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=info_modelo['parametros'],
    cv=5,
    scoring='accuracy',
    verbose=0,
    n_jobs=-1,
)

# Convertir los resultados a un DataFrame para una mejor visualización
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

# Imprimir el rendimiento de los modelos de clasificación
print("Rendimiento de los modelos de clasificación")
print(metricas.round(2))

# Imprimir el mejor modelo y su precisión
print('---------------------------------------------------')
print("MEJOR MODELO DE CLASIFICACIÓN")
print(f"Modelo: {mejor_modelo}")
print(f"Precisión: {mejor_precision:.2f}")

# Guardar el mejor modelo con pickle
with open('models/pipeline.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

c:\ML DS Learning\Hybridge\aprendizaje_supervisado_ai\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\ML DS Learning\Hybridge\aprendizaje_supervisado_ai\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Rendimiento de los modelos de clasificación
                                 Modelo  Precisión
6      Clasificador K-Nearest Neighbors       0.83
8                     Clasificador LGBM       0.82
3    Clasificador de Bosques Aleatorios       0.82
7                  Clasificador XGBoost       0.81
1   Clasificador de Vectores de Soporte       0.81
0                   Regresión Logística       0.81
5                 Clasificador AdaBoost       0.81
2     Clasificador de Árbol de Decisión       0.80
4     Clasificador de Gradient Boosting       0.80
9                            GaussianNB       0.79
10             Clasificador Naive Bayes       0.78
---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.83


## Refactorizar API

Finalmente, modificaremos el código de nuestro API para que funcione con este pipeline.

```python
from flask import Flask, request, jsonify
import pickle
import pandas as pd

app = Flask(__name__)

# Cargar el modelo guardado
with open('pipeline.pkl', 'rb') as archivo_modelo:
    modelo = pickle.load(archivo_modelo)

@app.route('/predecir', methods=['POST'])
def predecir():
    # Obtener los datos de la solicitud
    data = request.get_json()

    # Crear un DataFrame de pandas a partir del JSON
    input_data = pd.DataFrame([data])

    # Hacer la predicción usando el modelo que tiene el pipeline que hará la transformación
    prediccion = modelo.predict(input_data)
    
    # Devolver la predicción como JSON
    output = {'Survived': int(prediccion[0])}
    
    return jsonify(output)

if __name__ == '__main__':
    app.run(debug=True)